# 01 — Confirm the v3 G-SWAP prerequisite

Stage 0 passed. This notebook independently repeats the frozen v2 alpha-2,
all-position sentinel three times per item. It confirms that intervention
machinery has not drifted; alpha=2 is not selected for downstream use.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['HF_HOME'] = str(Path.home() / '.cache/huggingface')

metrics = json.loads((ROOT / 'results/metrics.json').read_text())
v3 = metrics['calibration_v3']
assert v3['gate_ledger']['stage0_reverify'] == 'PASS'
assert v3['gate_ledger']['stage3_science'] == 'PROHIBITED'
workspace_layers = v3['protocol']['workspace_layers']

from src.jlens_iface import load_published_lens
from src.model_utils import load_model
from src.v2_repair import MODEL_ID

bundle = load_model(MODEL_ID)
lens = load_published_lens(MODEL_ID)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [2]:
from src.v3_reverify import run_stage1_confirm

stage1 = run_stage1_confirm(bundle, lens, workspace_layers=workspace_layers)
for row in stage1['g_swap']['rows']:
    print({
        'item': row['name'],
        'swap': f"{row['source']}->{row['target']}",
        'clean_top': row['clean_top'],
        'edited_top': row['edited_top'],
        'clean_M': row['clean_metric'],
        'edited_M': row['edited_metric'],
        'repeat_error': row['repeat_max_abs_logit_error'],
        'pass': row['pass'],
    })
print('G-SWAP', stage1['status'])

{'item': 'spider-legs', 'swap': ' spider->ant', 'clean_top': '8', 'edited_top': '6', 'clean_M': 6.5, 'edited_M': -6.5, 'repeat_error': 0.0, 'pass': True}
{'item': 'animal-legs-buffalo2', 'swap': ' buffalo-> spider', 'clean_top': ' four', 'edited_top': ' eight', 'clean_M': 3.5625, 'edited_M': -4.0, 'repeat_error': 0.0, 'pass': True}
{'item': 'chem-photosynthesis-Z', 'swap': ' oxygen-> nitrogen', 'clean_top': '8', 'edited_top': '7', 'clean_M': 5.375, 'edited_M': -5.25, 'repeat_error': 0.0, 'pass': True}
G-SWAP PASS


In [3]:
from src.v3_reverify import persist_stage1

metrics = persist_stage1(stage1)
v3 = metrics['calibration_v3']
assert v3['gate_ledger']['g_swap'] == stage1['status']
assert v3['gate_ledger']['stage3_science'] == 'PROHIBITED'
print(json.dumps({
    'g_swap': v3['gate_ledger']['g_swap'],
    'g_alpha': v3['gate_ledger']['g_alpha'],
    'next': '015_alpha_sweep' if stage1['status'] == 'PASS' else '08_report',
}, indent=2))

{
  "g_swap": "PASS",
  "g_alpha": "PENDING",
  "next": "015_alpha_sweep"
}


In [4]:
import gc
import torch

del stage1, metrics, lens, bundle
gc.collect()
torch.cuda.empty_cache()
print('Notebook 01 complete. Science remains prohibited pending G-ALPHA.')

Notebook 01 complete. Science remains prohibited pending G-ALPHA.
